This notebook can be used to verify the dataframes displayed values in Napari. 

In [1]:
import napari
import pandas as pd
from pathlib import Path
import numpy as np
from skimage import io
viewer = napari.Viewer()
pd.set_option('display.max_columns', None)

In [4]:
df = pd.read_csv('Mara_budSegmentationData20260311.csv')
df = df.drop('Unnamed: 0',axis=1)

Construct unique identifier used to select cells for displaying in Napari.

In [5]:
df['fov']=df['source_folder'].str[-2:]
df['uid'] = (
    df['strain'].astype(str) + '_' +
    df['replicate'].astype(str) + '_' +
    df['image_cell_id'].astype(str) + '_' +
    df['fov'].astype(str)
)

In [7]:
#showcase only relevant columns
showcase_df = df[['uid','image_cell_id', 'mRNA', 'area',
       'area_bud', 'spots', 'dense_regions', 'decomposed_RNAs',
       'tx', 'nascent_RNAs', 'total_RNAs', 'spots_bud', 'dense_regions_bud',
       'decomposed_RNAs_bud', 'tx_bud', 'nascent_RNAs_bud',
       'total_RNAs_bud', 'mother_nuclei_count','spots_mother', 'total_RNAs_mother',
      'replicate','condition + strain']]

Look for dataframe entries to visualize in Napari.

In [22]:
showcase_df[showcase_df['dense_regions']>10]

,uid,image_cell_id,mRNA,area,area_bud,spots,dense_regions,decomposed_RNAs,tx,nascent_RNAs,total_RNAs,spots_bud,dense_regions_bud,decomposed_RNAs_bud,tx_bud,nascent_RNAs_bud,total_RNAs_bud,mother_nuclei_count,spots_mother,total_RNAs_mother,replicate,condition + strain
583,YET563_BR3_199_01,199,SUN4,2559.0,0.0,25,18,71,4,19,78,0,0,0,0,0,0,1.0,25,78,BR3,ΔSHE2 YET563
943,YET563_BR3_130_02,130,SRL1,4536.0,0.0,46,16,46,0,0,76,0,0,0,0,0,0,1.0,46,76,BR3,ΔSHE2 YET563
999,YET563_BR3_203_02,203,SRL1,4105.0,0.0,47,11,39,2,9,75,0,0,0,0,0,0,1.0,47,75,BR3,ΔSHE2 YET563
1007,YET563_BR3_217_02,217,SRL1,5455.0,0.0,51,11,36,3,12,76,0,0,0,0,0,0,1.0,51,76,BR3,ΔSHE2 YET563
1013,YET563_BR3_226_02,226,SRL1,4359.0,0.0,41,12,37,1,2,66,0,0,0,0,0,0,1.0,41,66,BR3,ΔSHE2 YET563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35806,YET916_BR1_163_31,163,SRL1,4037.0,0.0,43,14,37,2,5,66,0,0,0,0,0,0,1.0,43,66,BR1,ΔSSD1 YET916
35813,YET916_BR1_226_31,226,SRL1,9980.0,1861.0,79,16,57,1,3,120,19,15,54,0,0,58,1.0,60,62,BR1,ΔSSD1 YET916
36401,YET921_BR1_374_41,374,ASH1CLB2,4568.0,1818.0,23,11,68,7,33,80,17,10,61,6,26,68,1.0,6,12,BR1,ΔSEC72 YET921
36714,YET921_BR1_141_41,141,SUN4,2833.0,0.0,24,15,43,3,14,52,0,0,0,0,0,0,1.0,24,52,BR1,ΔSEC72 YET921


Use uid to select cell.

In [23]:
uid = 'YET563_BR3_199_01'	
rows = df[df['uid']==uid]
basepath = Path(rows['source_folder'].values[0])

# Get bounding box coordinates
y0 = int(rows['bbox-0'].iloc[0])
x0 = int(rows['bbox-1'].iloc[0])
y1 = int(rows['bbox-2'].iloc[0])
x1 = int(rows['bbox-3'].iloc[0])
showcase_df[showcase_df['uid']==uid]

,uid,image_cell_id,mRNA,area,area_bud,spots,dense_regions,decomposed_RNAs,tx,nascent_RNAs,total_RNAs,spots_bud,dense_regions_bud,decomposed_RNAs_bud,tx_bud,nascent_RNAs_bud,total_RNAs_bud,mother_nuclei_count,spots_mother,total_RNAs_mother,replicate,condition + strain
119,YET563_BR3_199_01,199,ASH1CLB2,2559.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,1.0,0,0,BR3,ΔSHE2 YET563
351,YET563_BR3_199_01,199,SRL1,2559.0,0.0,5,0,0,0,0,5,0,0,0,0,0,0,1.0,5,5,BR3,ΔSHE2 YET563
583,YET563_BR3_199_01,199,SUN4,2559.0,0.0,25,18,71,4,19,78,0,0,0,0,0,0,1.0,25,78,BR3,ΔSHE2 YET563


Show cell in napari.

In [26]:
spot_paths = list(basepath.glob('*spots.npy'))
masks_paths = list(Path(basepath,'corrected').glob('*.tif'))

ASH1CLB2 = np.amax(io.imread(Path(basepath,'ASH1CLB2.tif')),axis=0)
SRL1 = np.amax(io.imread(Path(basepath,'SRL1.tif')),axis=0)
SUN4 = np.amax(io.imread(Path(basepath,'SUN4.tif')),axis=0)
DAPI = np.amax(io.imread(Path(basepath,'DAPI.tif')),axis=0)

# Define rectangle as 2D array: [[y0, x0], [y0, x1], [y1, x1], [y1, x0]]
rectangle = [[y0, x0], [y0, x1], [y1, x1], [y1, x0]]
viewer.add_shapes([rectangle], shape_type='rectangle', edge_color='red',blending='additive',edge_width=4, face_color='transparent', name=f'cell_{uid}')

for spot_path in spot_paths[:3]:
    name = spot_path.name.split('_')[0] 
    viewer.add_points(np.load(spot_path)[:,1:3],name=spot_path.stem, visible=False, face_color='blue')
    if name == 'ASH1CLB2':
        viewer.add_image(ASH1CLB2, name='ASH1CLB2', blending='additive', visible=False, contrast_limits=[0,12000])
    elif name == 'SRL1':
        viewer.add_image(SRL1, name='SRL1', blending='additive', visible=False, contrast_limits=[0,12000])
    elif name == 'SUN4':
        viewer.add_image(SUN4, name='SUN4', blending='additive', visible=True ,contrast_limits=[0,12000])

viewer.add_image(DAPI, name='DAPI', blending='additive', visible=False ,contrast_limits=[0,25000])
        
for masks_path in masks_paths:
    viewer.add_labels(io.imread(masks_path),name=masks_path.stem)

Clear all layers.

In [25]:
viewer.layers.clear()